<a href="https://colab.research.google.com/github/ushandissanayaka/Currency-App/blob/main/NoteManageRAGApplication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. Install modern 2026-compliant AI libraries
!pip install -q streamlit langchain-ollama langchain-chroma pypdf

# Install zstd, a required dependency for Ollama extraction
!apt-get update -y
!apt-get install -y zstd

# 2. Install the Ollama execution framework into Colab's Linux container
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Initialize the Ollama background service daemon
import subprocess
import time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

# 4. Pull down the highly efficient models
!ollama pull llama3.2:1b
!ollama pull nomic-embed-text
print("✨ Environment successfully initialized! All models are loaded.")

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,713 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,006 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,301 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,183 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:13 http://archive.ubuntu.com/ubuntu j

In [4]:
%%writefile app.py
import streamlit as st
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, OllamaLLM
from langchain_chroma import Chroma
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

st.set_page_config(page_title="UniNotes Matrix Assistant", layout="wide")

st.title("🎓 UniVault: Year & Module Structured Academic RAG")
st.write("An isolated knowledge management system sorting university notes by Year and Module.")

# Root directory path allocation
BASE_DB_DIR = "./uni_knowledge_base"
os.makedirs(BASE_DB_DIR, exist_ok=True)

# Split screen workspace mapping
sidebar, main_chat = st.columns([1, 2])

with sidebar:
    st.header("📥 Document Ingestion Control")

    # Year Selection Dropdown
    selected_year = st.selectbox("Select Academic Year:", ["Year 1", "Year 2", "Year 3", "Year 4"])

    # Module Entry String Input
    module_code = st.text_input("Enter Module Code (e.g., CS101, EE302):", value="CS101").strip().upper()

    # File Uploader Target
    uploaded_file = st.file_uploader(f"Upload PDF Notes to {selected_year} -> {module_code}", type="pdf")

    if uploaded_file and module_code:
        # Build the structured file path directory
        target_db_path = os.path.join(BASE_DB_DIR, selected_year, module_code)

        with st.spinner("Processing document and generating local vectors..."):
            temp_file_name = f"temp_{module_code}.pdf"
            with open(temp_file_name, "wb") as f:
                f.write(uploaded_file.getbuffer())

            # Extract and chunk document texts
            loader = PyPDFLoader(temp_file_name)
            docs = loader.load()
            text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=120)
            chunks = text_splitter.split_documents(docs)

            # Generate vectors offline via Ollama Embeddings
            embeddings = OllamaEmbeddings(model="nomic-embed-text")

            # Persist directly into the exact nested folder directory structure
            vector_store = Chroma.from_documents(
                documents=chunks,
                embedding=embeddings,
                persist_directory=target_db_path
            )

            if os.path.exists(temp_file_name):
                os.remove(temp_file_name)

        st.success(f"Successfully added to storage vault: {selected_year} ➔ {module_code}")

with main_chat:
    st.header("💬 Contextual Academic Chat")

    st.markdown("### Select Context Target")
    chat_year = st.selectbox("Target Academic Year:", ["Year 1", "Year 2", "Year 3", "Year 4"], key="chat_yr")
    chat_module = st.text_input("Target Module Code:", value="CS101", key="chat_mod").strip().upper()

    active_target_path = os.path.join(BASE_DB_DIR, chat_year, chat_module)

    # Verification check to see if the targeted data path has valid files
    if os.path.exists(active_target_path) and len(os.listdir(active_target_path)) > 0:
        user_query = st.text_input(f"Ask any question regarding {chat_year} - {chat_module} context:")

        if user_query:
            with st.spinner("Analyzing target documentation structure..."):
                # Load corresponding index matrix directory
                embeddings = OllamaEmbeddings(model="nomic-embed-text")
                vector_store = Chroma(persist_directory=active_target_path, embedding_function=embeddings)
                retriever = vector_store.as_retriever(search_kwargs={"k": 3})

                # Setup localized language engine variables
                llm = OllamaLLM(model="llama3.2:1b", temperature=0.1)

                system_prompt = (
                    "You are a helpful university tutor. Answer the student's question clearly, "
                    "organizing steps if necessary. Base your answer strictly on the provided context "
                    "from their course documents below. If the information is missing, state clearly "
                    "that you cannot find it in the uploaded lecture notes.\n\n"
                    "Course Context Data:\n{context}"
                )
                prompt = ChatPromptTemplate.from_messages([
                    ("system", system_prompt),
                    ("human", "{input}"),
                ])

                # Execution of the active document chain processing
                document_chain = create_stuff_documents_chain(llm, prompt)
                rag_chain = create_retrieval_chain(retriever, document_chain)

                response = rag_chain.invoke({"input": user_query})

                st.markdown("#### 📝 System Answer:")
                st.info(response["answer"])
    else:
        st.warning(f"The vector storage bank for {chat_year} -> {chat_module} is currently empty. Upload files in the sidebar to begin.")

Overwriting app.py


In [ ]:
# 1. Print out your unique local gateway execution password IP
print("👇 COPY THIS IP ADDRESS INTERFACE PASSWORD:")
!curl ipv4.icanhazip.com
print("-" * 50)

# 2. Deploy Streamlit interface and expose via localtunnel proxy port mappings
!streamlit run app.py & npx localtunnel --port 8501

👇 COPY THIS IP ADDRESS INTERFACE PASSWORD:
35.240.169.183
--------------------------------------------------
⠙

⠹⠸⠼⠴⠦Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 2026-06-09 17:47:06.224 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.240.169.183:8501

